# Archived SAGA convergence probe

This incomplete attempt is not used to select the final penalty model. Some weakly regularized fits hit max_iter=100000 despite retries; the main experiment switches to a full-batch convex optimizer. The executed error and completed probe fits are retained for transparency.

# L1, Elastic Net, and matched-solver L2 logistic regression

Compare penalty types while keeping original DBSCAN targets/raw weights, normalized fitting-weight total 157 per reason, 150 embeddings/scaler, class weighting, cutoff 0.5, and highest-probability fallback fixed. The incumbent is the saved LBFGS/L2 model at C=1000 (sample F1 0.910256, macro F1 0.811421).

Predeclared grid: **C=0.1,1,10,100,500,1000,2000** and **L1 fraction=0,0.25,0.50,0.75,1**. Fractions 0 and 1 are pure L2 and L1; intermediate fractions are Elastic Net. This gives **35 complete five-output configurations / 175 binary fits**, using a shared penalty/C within each configuration. The earlier per-reason-C sweep is a separate experiment.

Use SAGA for every grid setting, including matched-solver L2 controls. Start at tol=1e-5 with max_iter=100000 and seed=42. Require no convergence warning and normalized KKT residual <=1e-4; if needed retry at tol=1e-6 and then 1e-7. Every retry is governed by training convergence, never validation score. Never rank an unconverged fit. Record objective, KKT residual, iteration count, nonzero coefficients, and attempts. If the C=1000 SAGA/L2 control differs in labels from the incumbent, check tightly converged LBFGS/L2 to diagnose the numerical difference; this is not another penalty setting.

With randomly rotated embeddings, sparsity is coordinate-dependent, so L1 is not presumed superior. Keep the current model on score ties. No new features, pseudo-label generation, validation-label refit, Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.special import expit
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
FEATURES=[f"emb_{i}" for i in range(150)]
C_GRID=[0.1,1.0,10.0,100.0,500.0,1000.0,2000.0]
L1_FRACTIONS=[0.0,0.25,0.50,0.75,1.0]
TOLERANCES=[1e-5,1e-6,1e-7]
MAX_ITER=100000
KKT_LIMIT=1e-4
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/additional_methods"
OUT=DATA/"experiments/logistic_penalties/saga_probe"
for folder in ["datasets","reason_models","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv");valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
original=train[REASONS].to_numpy(dtype=int);truth=valid[REASONS].to_numpy(dtype=int)
assert len(train)==1171 and known.sum()==157 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
source_model_path=SOURCE/source_record["model_file"];source_dataset_path=SOURCE/"datasets/dbscan_m5_q25.csv"
source_model=joblib.load(source_model_path)
fixed=pd.read_csv(source_dataset_path,float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_source.csv").write_bytes(source_dataset_path.read_bytes())
scaler=source_model["scaler"];assert scaler.n_samples_seen_==157
XS=scaler.transform(train[FEATURES].to_numpy());VS=scaler.transform(valid[FEATURES].to_numpy())
NORMALIZED_W=W*157/W.sum(axis=0)
weights_frame=train[["ID"]].copy()
for j,r in enumerate(REASONS):weights_frame[f"effective_weight_{r}"]=NORMALIZED_W[:,j]
weights_frame.to_csv(OUT/"datasets/normalized_fitting_weights.csv",index=False)
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",source_model_path,source_dataset_path,
                 SOURCE/source_record["submission_file"],SOURCE/"labeling_models/dbscan_m5_q25.joblib",SOURCE/"submission_manifest.json"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("35 penalty/C configurations; all data, scaler, source labels/weights, and prediction rules frozen.")


35 penalty/C configurations; all data, scaler, source labels/weights, and prediction rules frozen.


In [2]:
def predict_model(model,raw):
    scaled=model["scaler"].transform(np.asarray(raw))
    return np.column_stack([est.predict_proba(scaled)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int);empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty


def training_stationarity(estimator,X,y,weights,C,ratio):
    coefficient=estimator.coef_[0];intercept=float(estimator.intercept_[0]);margin=X@coefficient+intercept
    residual=weights*(expit(margin)-y);total=weights.sum()
    smooth=(X.T@residual+(1-ratio)*coefficient/C)/total
    l1_strength=ratio/(C*total)
    nonzero=np.abs(coefficient)>1e-10
    stationarity=np.maximum(np.abs(smooth)-l1_strength,0)
    stationarity[nonzero]=np.abs(smooth[nonzero]+l1_strength*np.sign(coefficient[nonzero]))
    intercept_residual=abs(float(residual.sum()/total))
    objective=float((np.dot(weights,np.logaddexp(0,margin)-y*margin)+
                    (0.5*(1-ratio)*np.dot(coefficient,coefficient)+ratio*np.abs(coefficient).sum())/C)/total)
    return {"kkt_residual":max(float(stationarity.max()),intercept_residual),"objective":objective,
            "intercept_gradient":intercept_residual,"nonzero_coefficients":int(np.count_nonzero(coefficient)),
            "coefficient_norm":float(np.linalg.norm(coefficient))}

reference_probability=predict_model(source_model,valid[FEATURES].to_numpy())
reference_prediction,_=label_predictions(reference_probability)
reference_sample=f1_score(truth,reference_prediction,average="samples",zero_division=0)
reference_macro=f1_score(truth,reference_prediction,average="macro",zero_division=0)
np.testing.assert_allclose(reference_sample,source_record["validation_sample_f1"],atol=1e-14)
ROWS=[];FIT_ROWS=[];ATTEMPTS=[];MODEL_CACHE={};PROB_CACHE={}


def record_candidate(name,model,ratio,C,stage,fit_records):
    probabilities=predict_model(model,valid[FEATURES].to_numpy());prediction,empty=label_predictions(probabilities)
    sample=f1_score(truth,prediction,average="samples",zero_division=0)
    macro=f1_score(truth,prediction,average="macro",zero_division=0)
    row={"experiment":name,"stage":stage,"penalty_type":"L2" if ratio==0 else ("L1" if ratio==1 else "ElasticNet"),
         "l1_fraction":ratio,"C":C,"solver":model["estimators"][0].solver,"sample_f1":sample,"macro_f1":macro,
         **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,prediction,average=None,zero_division=0))),
         "delta_vs_incumbent":sample-reference_sample,"fallback_trials":int(empty.sum()),
         "changed_trials_vs_incumbent":int((prediction!=reference_prediction).any(axis=1).sum()),
         "total_nonzero_coefficients":sum(np.count_nonzero(e.coef_) for e in model["estimators"]),
         "mean_nonzero_per_reason":float(np.mean([np.count_nonzero(e.coef_) for e in model["estimators"]])),
         "maximum_kkt_residual":max(v["kkt_residual"] for v in fit_records) if fit_records else np.nan,
         "maximum_iterations":max(int(e.n_iter_.max()) for e in model["estimators"]),"kaggle_score":np.nan}
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=prediction[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=2*(truth*prediction).sum(axis=1)/(truth.sum(axis=1)+prediction.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
    ROWS.append(row);MODEL_CACHE[name]=model;PROB_CACHE[name]=probabilities
    pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
    return row

record_candidate("incumbent_lbfgs_l2_C1000",source_model,0.0,1000.0,"incumbent",[])


def run_penalty(ratio):
    penalty="L2" if ratio==0 else ("L1" if ratio==1 else "ElasticNet")
    for C in C_GRID:
        name=f"{penalty}_ratio{ratio:g}_C{C:g}".replace(".","p")
        estimators=[];fit_records=[]
        with threadpool_limits(limits=1):
            for j,reason in enumerate(REASONS):
                included=W[:,j]>0;x=XS[included];y=Y[included,j];weights=NORMALIZED_W[included,j]
                accepted=False
                for tolerance in TOLERANCES:
                    estimator=LogisticRegression(C=C,l1_ratio=ratio,solver="saga",max_iter=MAX_ITER,tol=tolerance,
                                                 class_weight=None,random_state=42)
                    start=time.monotonic()
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter("always",ConvergenceWarning)
                        estimator.fit(x,y,sample_weight=weights)
                    warned=any(issubclass(w.category,ConvergenceWarning) for w in caught)
                    diagnostics=training_stationarity(estimator,x,y,weights,C,ratio)
                    ATTEMPTS.append({"experiment":name,"reason":reason,"tol":tolerance,"seconds":time.monotonic()-start,
                                     "iterations":int(estimator.n_iter_.max()),"convergence_warning":warned,**diagnostics})
                    if not warned and diagnostics["kkt_residual"]<=KKT_LIMIT:
                        accepted=True;break
                if not accepted:raise RuntimeError(f"Unconverged model excluded: {name}, {reason}, {diagnostics}")
                estimator_name=f"{name}_{reason}"
                joblib.dump(estimator,OUT/"reason_models"/f"{estimator_name}.joblib",compress=3)
                fit_row={"experiment":name,"reason":reason,"penalty_type":penalty,"C":C,"l1_fraction":ratio,
                         "tol":tolerance,"iterations":int(estimator.n_iter_.max()),"training_rows":int(included.sum()),
                         "fitting_weight_total":float(weights.sum()),**diagnostics}
                FIT_ROWS.append(fit_row);fit_records.append(fit_row);estimators.append(estimator)
        model={**source_model,"estimators":estimators,"parameter":C,"l1_fraction":ratio,"penalty_type":penalty,
               "solver":"saga","source_dataset":"datasets/fixed_source.csv","normalized_weight_file":"datasets/normalized_fitting_weights.csv"}
        row=record_candidate(name,model,ratio,C,"penalty_grid",fit_records)
        pd.DataFrame(FIT_ROWS).to_csv(OUT/"fit_audit.csv",index=False)
        pd.DataFrame(ATTEMPTS).to_csv(OUT/"optimization_attempts.csv",index=False)
        print(f"{name}: sample F1={row['sample_f1']:.6f}, macro F1={row['macro_f1']:.6f}, "
              f"nonzero={row['total_nonzero_coefficients']}/750, max KKT={row['maximum_kkt_residual']:.2g}",flush=True)


In [3]:
run_penalty(0.0)


L2_ratio0_C0p1: sample F1=0.841880, macro F1=0.599954, nonzero=750/750, max KKT=9.5e-05


L2_ratio0_C1: sample F1=0.858974, macro F1=0.695743, nonzero=750/750, max KKT=9.3e-05


L2_ratio0_C10: sample F1=0.867521, macro F1=0.718678, nonzero=750/750, max KKT=8.9e-05


L2_ratio0_C100: sample F1=0.880342, macro F1=0.734238, nonzero=750/750, max KKT=6.4e-05


RuntimeError: Unconverged model excluded: L2_ratio0_C500, Safety, {'kkt_residual': 1.2560101397997112e-05, 'objective': 0.00010623115621042008, 'intercept_gradient': 1.2560101397997112e-05, 'nonzero_coefficients': 150, 'coefficient_norm': 3.5152650261796756}

In [ ]:
run_penalty(0.25)


In [ ]:
run_penalty(0.5)


In [ ]:
run_penalty(0.75)


In [ ]:
run_penalty(1.0)


In [ ]:
grid=pd.DataFrame(ROWS).query("stage=='penalty_grid'")
assert len(grid)==35 and len(FIT_ROWS)==175
solver_control=MODEL_CACHE["L2_ratio0_C1000"]
solver_prediction,_=label_predictions(PROB_CACHE["L2_ratio0_C1000"])
SOLVER_DIAGNOSTIC=False
if not np.array_equal(solver_prediction,reference_prediction):
    SOLVER_DIAGNOSTIC=True
    estimators=[];records=[]
    with threadpool_limits(limits=1):
        for j,reason in enumerate(REASONS):
            included=W[:,j]>0
            estimator=clone(source_model["estimators"][j]).set_params(tol=1e-9,max_iter=20000)
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(XS[included],Y[included,j],sample_weight=NORMALIZED_W[included,j])
            diagnostic=training_stationarity(estimator,XS[included],Y[included,j],NORMALIZED_W[included,j],1000,0)
            assert diagnostic["kkt_residual"]<=KKT_LIMIT
            records.append(diagnostic);estimators.append(estimator)
    record_candidate("diagnostic_lbfgs_tight_l2_C1000",{**source_model,"estimators":estimators},0,1000,"solver_diagnostic",records)
comparison=pd.DataFrame(ROWS)
matched=grid.query("l1_fraction==0").set_index("C").sample_f1.to_dict()
comparison["delta_vs_same_C_saga_L2"]=[row.sample_f1-matched[row.C] for row in comparison.itertuples(index=False)]
comparison.to_csv(OUT/"comparison.csv",index=False)
ranked=comparison.assign(is_incumbent=comparison.stage.eq("incumbent"),distance_C=abs(np.log10(comparison.C/1000))).sort_values(
    ["sample_f1","is_incumbent","l1_fraction","distance_C","C"],ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_penalty=ranked.groupby("penalty_type",sort=False).head(1)
best_by_penalty.to_csv(OUT/"best_by_penalty.csv",index=False)
print("Matched-solver C=1000 control changed labels:",SOLVER_DIAGNOSTIC)
print("Selected:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(best_by_penalty[["penalty_type","l1_fraction","C","solver","sample_f1","macro_f1","delta_vs_incumbent","delta_vs_same_C_saga_L2"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid[FEATURES].to_numpy())
for row in comparison.itertuples(index=False):
    model=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy();prediction=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_model(model,valid[FEATURES].to_numpy()),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],prediction)
    np.testing.assert_allclose(f1_score(truth,prediction,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,prediction,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
frame=pd.read_csv(OUT/"datasets/fixed_source.csv",float_precision="round_trip")
weights=pd.read_csv(OUT/"datasets/normalized_fitting_weights.csv",float_precision="round_trip")
assert frame.ID.tolist()==train.ID.tolist()==weights.ID.tolist()
np.testing.assert_array_equal(frame[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(frame[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
scaled=scaler.transform(frame[FEATURES].to_numpy());replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,reason in enumerate(REASONS):
        w=weights[f"effective_weight_{reason}"].to_numpy();included=w>0
        np.testing.assert_allclose(w,NORMALIZED_W[:,j],rtol=1e-12,atol=1e-12)
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],Y[included,j],sample_weight=w[included])
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid[FEATURES].to_numpy()),selected_probability,rtol=1e-6,atol=1e-8)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid[FEATURES].to_numpy()))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("All accepted grid fits meet convergence/KKT limits; saved scores/models and independent selected-model refit verified.")


In [ ]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"penalty_type":best.penalty_type,"l1_fraction":float(best.l1_fraction),"C":float(best.C),
               "validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"C_grid":C_GRID,"l1_fraction_grid":L1_FRACTIONS,"grid_configurations":35,"binary_grid_fits":175,
          "solver":"saga","max_iter":MAX_ITER,"tolerances_by_retry":TOLERANCES,"kkt_acceptance_limit":KKT_LIMIT,
          "tighter_LBFGS_diagnostic_run":SOLVER_DIAGNOSTIC,"total_complete_configurations":len(comparison),
          "features":FEATURES,"classifier_scaler_fit_rows":157,"source_targets_changed":False,"source_raw_weights_changed":False,
          "fitting_weight_total_per_reason":157,"class_weight":None,"cutoff":0.5,"fallback":True,
          "validation_rows":78,"validation_used_for_fit":False,"final_validation_label_refit":False,
          "reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "best_penalty_type":best.penalty_type,"best_C":float(best.C),"best_l1_fraction":float(best.l1_fraction),
          "leader_changed":bool(best.sample_f1>reference_sample+1e-12),"candidate":candidate,"protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "selection":"sample F1; ties retain incumbent, then favor smaller L1 fraction and C closest to 1000",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(rows):
    lines=["| Penalty | L1 fraction | C | Solver | Sample F1 | Macro F1 | Nonzero coefficients /750 | Change vs incumbent | Change vs matched L2 |",
           "|---|---:|---:|---|---:|---:|---:|---:|---:|"]
    for row in rows.itertuples(index=False):
        lines.append(f"| {row.penalty_type} | {row.l1_fraction:g} | {row.C:g} | {row.solver} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.total_nonzero_coefficients} | {row.delta_vs_incumbent:+.6f} | {row.delta_vs_same_C_saga_L2:+.6f} |")
    return "\n".join(lines)
lines=["# Logistic penalty-type comparison","",
       "35 grid settings cross C=0.1/1/10/100/500/1000/2000 with L1 fractions 0/0.25/0.50/0.75/1. "
       "Zero is L2, one is L1, intermediate values are Elastic Net. Every grid setting uses SAGA. "
       "The frozen original LBFGS/L2 model is retained as incumbent, with same-solver L2 controls at each C.","",
       "All 175 binary grid fits require no convergence warning and normalized KKT residual <=1e-4. "
       "Training convergence alone governs tolerance retries (1e-5, then 1e-6, then 1e-7) with max_iter=100000. "
       "Objectives, stationarity residuals, iterations, attempts, and nonzero coefficients are recorded. "
       f"A tighter-LBFGS solver diagnostic was run: **{SOLVER_DIAGNOSTIC}**.","",
       f"Incumbent sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best by penalty family","",table(best_by_penalty),"",
       "## All configurations","",table(ranked),"",
       "## Controls, verification, and interpretation","",
       "The exact 150-dimensional embedding/scaler, DBSCAN targets/raw weights, normalized fitting-weight total 157, "
       "class_weight=None, cutoff 0.5, and highest-probability fallback remain fixed. No new features or pseudo-labels "
       "are generated. The compared C and penalty are shared across the five reason classifiers within each configuration. "
       "Validation labels never enter fitting and no final train+validation refit is performed.","",
       "Source targets/weights and protected artifacts retain their hashes. Every saved complete model reloads with "
       "matching probabilities and independently rescored sample/macro F1. The selected model independently refits "
       "from persisted fixed data and normalized weights with matching predictions. Penalty effects are compared "
       "with same-C, same-solver L2 controls; a changed solver alone is not attributed to L1/Elastic Net.","",
       "The embeddings were randomly rotated, so sparse coordinate selection is not presumed helpful. Results are "
       "exploratory selections on the repeatedly consulted 78 validation trials and do not establish independent test "
       "improvement or exhaust per-reason penalties/C combinations.","",
       "datasets/ preserves original data/weights and exact normalized weights; reason_models/ saves all 175 binary "
       "fits; models/ and validation_predictions/ preserve every full configuration. fit_audit.csv, optimization_attempts.csv, "
       "comparison.csv, best_by_penalty.csv, and settings.json preserve convergence, sparsity, scores, and checksums.","",
       "No Kaggle submission, commit, or tag was made.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No setting improves primary sample F1; retain the existing L2 model and its prepared Kaggle file.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print(table(best_by_penalty))
print("Leader changed:",settings['leader_changed'],"Candidate:",candidate)
